# Exercise 1. 피싱 웹사이트 분류 — 완전 실습
**목표:** 데이터 점검 → 전처리 → 6개 분류 모델 비교 → 혼동행렬·ROC/PR 곡선 → 교차검증 → 특성 중요도 → 결과 저장

정답은 `label`이며 `normal=0`, `phishing=1`로 사용합니다. 보안 탐지에서는 피싱을 정상으로 놓치는 **FN(미탐)**과 정상 사이트를 피싱으로 판단하는 **FP(오탐)**을 구분해야 합니다.

In [1]:
import warnings, os, glob, numpy as np, pandas as pd, matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
from IPython.display import display
RANDOM_STATE=42
np.random.seed(RANDOM_STATE)
pd.set_option("display.max_columns",100)
print("환경 준비 완료")

환경 준비 완료


In [2]:
EXPECTED_KEYWORD="phishing_websites"
paths=(glob.glob("/content/*.csv")+glob.glob("/content/drive/MyDrive/**/*.csv",recursive=True)
       +glob.glob("./*.csv")+glob.glob("/mnt/data/*.csv"))
matched=[p for p in paths if EXPECTED_KEYWORD.lower() in os.path.basename(p).lower()]
if not matched:
    from google.colab import files
    print("CSV 파일을 업로드하세요.")
    uploaded=files.upload()
    matched=list(uploaded.keys())
file_path=matched[0]
df=pd.read_csv(file_path)
print("파일:",file_path,"크기:",df.shape)
display(df.head())

CSV 파일을 업로드하세요.


Saving Exercise1. phishing_websites_sample.csv to Exercise1. phishing_websites_sample.csv
파일: Exercise1. phishing_websites_sample.csv 크기: (1000, 31)


,having_IP_Address,URL_Length,Shortining_Service,having_At_Symbol,double_slash_redirecting,Prefix_Suffix,having_Sub_Domain,SSLfinal_State,Domain_registeration_length,Favicon,port,HTTPS_token,Request_URL,URL_of_Anchor,Links_in_tags,SFH,Submitting_to_email,Abnormal_URL,Redirect,on_mouseover,RightClick,popUpWidnow,Iframe,age_of_domain,DNSRecord,web_traffic,Page_Rank,Google_Index,Links_pointing_to_page,Statistical_report,label
0,-1,-1,-1,1,-1,-1,-1,-1,1,1,1,1,-1,0,-1,-1,1,1,0,1,-1,1,1,-1,1,-1,1,1,0,-1,phishing
1,1,-1,1,1,1,-1,1,-1,1,1,1,1,-1,-1,0,-1,1,1,0,1,1,1,1,-1,-1,-1,-1,1,1,-1,phishing
2,1,-1,1,1,1,-1,0,0,1,1,1,1,-1,0,-1,-1,1,1,0,1,1,1,1,1,-1,-1,-1,1,1,1,phishing
3,1,-1,1,1,1,-1,-1,1,-1,1,1,1,1,0,0,-1,1,1,0,1,1,1,1,1,-1,0,-1,-1,1,1,phishing
4,1,-1,1,1,1,-1,1,1,-1,1,1,1,1,0,0,-1,1,1,0,1,1,1,1,-1,1,1,-1,1,0,1,normal


## 1. 데이터 품질 확인
결측치, 중복값, 자료형, 클래스 불균형을 확인합니다.

## 2. 전처리 및 데이터 분할
중복 제거, 결측치 최빈값 대치, 8:2 층화 분할을 수행합니다.

## 3. 모델 학습
로지스틱 회귀, KNN, 나이브베이즈, SVM, 의사결정나무, 랜덤포레스트를 같은 데이터로 비교합니다.

## 4. 평가지표
Accuracy, Precision, Recall, F1, ROC-AUC를 계산합니다. 미탐 방지가 중요하면 Recall, 오탐 억제가 중요하면 Precision을 중점적으로 봅니다.

## 5. 혼동행렬·분류 보고서

## 6. ROC 및 Precision-Recall 곡선